# MIMIC-IV Cohort Integrity Tests

15 tests on `mimiciv_cohort.parquet` (n=6,543) covering data integrity, clinical plausibility, LLM input quality, and benchmark balance.

**Run order:** top to bottom. Each cell is independent given setup is run first.

**Output convention:**
- ✅ PASS — within expected range
- ⚠️  WARN — atypical but possibly real
- ❌ FAIL — likely data error, investigate

## Setup

In [ ]:
"""Setup: load cohort and BigQuery client."""
import pandas as pd
import numpy as np
from pathlib import Path
from google.cloud import bigquery

PROJECT = "sds-ss-2026q2"
COHORT_PATH = Path.home() / "Desktop/HealthcareAI_Stanford/results/mimiciv_cohort.parquet"

df = pd.read_parquet(COHORT_PATH)
client = bigquery.Client(project=PROJECT)

# Convert string-cast datetimes back to pandas Timestamps for temporal tests
for c in ["admittime", "dischtime", "icu_intime", "icu_outtime"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

print(f"Cohort loaded: {len(df):,} patients, {len(df.columns)} columns")
print(f"BigQuery project: {PROJECT}")

# Helper to push cohort hadm_ids to BigQuery for joined tests
def cohort_hadm_csv():
    return ",".join(str(h) for h in df.hadm_id.unique())

def verdict(passed, warn=False):
    return "❌ FAIL" if not passed else ("⚠️  WARN" if warn else "✅ PASS")

## Test 1 — No Duplicate Patients
Every row must be a unique `subject_id`. Violations indicate the `ROW_NUMBER()` partition for first-stay-only failed.

In [ ]:
"""Test 1: every row is a unique patient (first ICU stay only)."""
total = len(df)
unique = df.subject_id.nunique()
dups = total - unique
print(f"Total rows:       {total:,}")
print(f"Unique patients:  {unique:,}")
print(f"Duplicates:       {dups}")
print(verdict(dups == 0))

## Test 2 — Temporal Coherence
All timestamps must satisfy: `admittime < dischtime`, `admittime <= icu_intime <= icu_outtime <= dischtime`. Even one violation is a linkage error.

In [ ]:
"""Test 2: temporal coherence across admit/discharge/ICU windows."""
v = df[(df.dischtime <= df.admittime)
     | (df.icu_outtime > df.dischtime)
     | (df.icu_intime < df.admittime)
     | (df.icu_outtime < df.icu_intime)]
print(f"Temporal violations: {len(v)}")
if len(v) > 0:
    print(v[["hadm_id","admittime","icu_intime","icu_outtime","dischtime"]].head())
print(verdict(len(v) == 0))

## Test 3 — Hospital LOS ≥ ICU LOS
Total hospital stay must be at least as long as the ICU stay it contains.

In [ ]:
"""Test 3: hospital LOS >= ICU LOS (with 0.01 day tolerance)."""
v = df[df.icu_los_days > df.hospital_los_days + 0.01]
print(f"LOS violations: {len(v)}  ({len(v)/len(df)*100:.2f}%)")
if len(v) > 0:
    print(v[["hadm_id","icu_los_days","hospital_los_days"]].head())
# A handful of violations can occur from rounding or same-day discharge — warn under 1%
print(verdict(len(v) == 0, warn=(0 < len(v) < len(df)*0.01)))

## Test 4 — CCI Spot-Check Re-validation
Recompute Charlson on a 500-patient random sample directly from `diagnoses_icd` and compare to the stored `cci_score`. Adaptation: original SQL referenced a `recomputed_cci` table that does not exist — we recompute inline using identical Quan 2005 regex logic.

In [ ]:
"""Test 4: re-validate CCI by recomputing from raw diagnoses for 500 random patients."""
sample = df.sample(min(500, len(df)), random_state=42)[["hadm_id","cci_score"]]
hadm_list = ",".join(str(h) for h in sample.hadm_id)

recompute_sql = f"""
WITH cci_flags AS (
  SELECT hadm_id,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^I2[12]|^I252') THEN 1 ELSE 0 END) AS mi,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^I50|^I43|^I110|^I130|^I132') THEN 1 ELSE 0 END) AS chf,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^I7[01]|^I73|^K551') THEN 1 ELSE 0 END) AS pvd,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^G4[56]|^H340|^I6') THEN 1 ELSE 0 END) AS cvd,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^F0[0-3]|^F051|^G30|^G311') THEN 1 ELSE 0 END) AS dementia,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^J4[0-7]|^J6[0-7]|^J684|^J701|^J703') THEN 1 ELSE 0 END) AS copd,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^M0[5689]|^M3[2-6]|^M45') THEN 1 ELSE 0 END) AS ctd,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^K2[5-8]') THEN 1 ELSE 0 END) AS pud,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^K7[034]|^B18') THEN 1 ELSE 0 END) AS mild_liver,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^E1[0-4][016890]') THEN 1 ELSE 0 END) AS dm_no_cc,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^G8[1-3]') THEN 1 ELSE 0 END) AS hemiplegia,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^N1[89]|^I120|^I131') THEN 1 ELSE 0 END) AS renal,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^E1[0-4][2-57]') THEN 1 ELSE 0 END) AS dm_cc,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^C[0-7][0-6]|^C8[1-58]|^C9[0-7]') AND NOT REGEXP_CONTAINS(icd_code, r'^C7[7-9]|^C80') THEN 1 ELSE 0 END) AS cancer,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^K704|^K721|^K76[567]|^I850') THEN 1 ELSE 0 END) AS severe_liver,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^C7[7-9]|^C80') THEN 1 ELSE 0 END) AS metastatic,
    MAX(CASE WHEN REGEXP_CONTAINS(icd_code, r'^B2[0-24]') THEN 1 ELSE 0 END) AS hiv
  FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd`
  WHERE icd_version = 10 AND hadm_id IN ({hadm_list})
  GROUP BY hadm_id
)
SELECT hadm_id,
  (mi+chf+pvd+cvd+dementia+copd+ctd+pud+mild_liver+dm_no_cc)
  + (hemiplegia+renal+dm_cc+cancer)*2 + (severe_liver)*3 + (metastatic+hiv)*6 AS cci_recomputed
FROM cci_flags
"""

recomp = client.query(recompute_sql).to_dataframe()
merged = sample.merge(recomp, on="hadm_id", how="left")
merged["delta"] = (merged.cci_score - merged.cci_recomputed).abs()
mismatches = merged[merged.delta > 0]
print(f"Sampled: {len(merged)}  Mismatches: {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches.head(10))
print(verdict(len(mismatches) == 0, warn=(len(mismatches) <= 5)))

## Test 5 — Age Distribution Sanity

In [ ]:
"""Test 5: age min/max/median in plausible ICU ranges."""
a = df.anchor_age
u18 = (a < 18).sum()
o105 = (a > 105).sum()
print(f"min={a.min()}  max={a.max()}  mean={a.mean():.1f}  median={a.median():.0f}")
print(f"under_18: {u18}   over_105: {o105}")
# MIMIC-IV anchor_age is shifted to 91 for patients >89 — so >91 is expected
print(verdict(u18 == 0))

## Test 6 — ICU LOS Distribution

In [ ]:
"""Test 6: ICU LOS distribution shape and outliers."""
l = df.icu_los_days
q = l.quantile([0.5, 0.95, 0.99])
print(f"min={l.min():.1f}  max={l.max():.1f}  mean={l.mean():.1f}")
print(f"p50={q[0.5]:.1f}  p95={q[0.95]:.1f}  p99={q[0.99]:.1f}")
print(f">60 days: {(l > 60).sum()}")
# Expected: median 3-5, p95 < 30
in_range = 3 <= q[0.5] <= 5 and q[0.95] < 30
print(verdict(in_range, warn=not in_range))

## Test 7 — Mortality Rate Plausibility
Expected 15–28% for complex multi-morbid ICU; outside this range may indicate cohort skew.

In [ ]:
"""Test 7: in-hospital mortality plausibility."""
deaths = int(df.hospital_expire_flag.sum())
rate = deaths / len(df) * 100
print(f"Deaths: {deaths:,} / {len(df):,}  =  {rate:.1f}%")
print(verdict(15 <= rate <= 28, warn=(10 <= rate < 15 or 28 < rate <= 35)))

## Test 8 — CCI vs Mortality (Monotonicity)
Mortality should rise monotonically: moderate < high < very_high. Non-monotonic = ICD-10 regex bug.

In [ ]:
"""Test 8: mortality must increase with complexity tier."""
order = ["moderate", "high", "very_high"]
g = df.groupby("complexity_tier").agg(n=("hadm_id","count"),
                                       deaths=("hospital_expire_flag","sum"))
g["mortality_pct"] = (g.deaths / g.n * 100).round(1)
g = g.reindex(order)
print(g)
rates = g.mortality_pct.values
monotonic = all(rates[i] <= rates[i+1] for i in range(len(rates)-1))
print(verdict(monotonic))

## Test 9 — Gender Distribution

In [ ]:
"""Test 9: gender split should not exceed 75% in either direction."""
g = df.gender.value_counts(normalize=True) * 100
print(g.round(1).to_string())
print(verdict(g.max() <= 75, warn=(60 < g.max() <= 75)))

## Test 10 — Discharge Note Length Distribution
Joins cohort to `mimiciv_note.discharge` on BigQuery.

In [ ]:
"""Test 10: discharge note character-length distribution."""
sql = f"""
SELECT LENGTH(text) AS chars
FROM `physionet-data.mimiciv_note.discharge` d
WHERE hadm_id IN ({cohort_hadm_csv()})
"""
r = client.query(sql).to_dataframe()
q = r.chars.quantile([0.10, 0.50, 0.90, 0.99])
stub = (r.chars < 500).sum()
giant = (r.chars > 100000).sum()
print(f"p10={int(q[0.1])}  p50={int(q[0.5])}  p90={int(q[0.9])}  p99={int(q[0.99])}")
print(f"stub_notes (<500 chars): {stub}")
print(f"giant_notes (>100K chars): {giant}")
print(verdict(5000 <= q[0.5] <= 15000, warn=True))

## Test 11 — Notes per Admission

In [ ]:
"""Test 11: how many discharge notes per admission?"""
sql = f"""
SELECT hadm_id, COUNT(*) AS n
FROM `physionet-data.mimiciv_note.discharge`
WHERE hadm_id IN ({cohort_hadm_csv()})
GROUP BY hadm_id
"""
r = client.query(sql).to_dataframe()
q = r.n.quantile([0.10, 0.50, 0.90])
print(f"p10={int(q[0.1])}  median={int(q[0.5])}  p90={int(q[0.9])}")
print(f"single-note admissions: {(r.n == 1).sum()}")
print(f"note-rich (>50): {(r.n > 50).sum()}")
print(verdict(True, warn=True))  # informational

## Test 12 — Lab Result Density

In [ ]:
"""Test 12: lab events per admission."""
sql = f"""
SELECT hadm_id, COUNT(*) AS n
FROM `physionet-data.mimiciv_3_1_hosp.labevents`
WHERE hadm_id IN ({cohort_hadm_csv()})
GROUP BY hadm_id
"""
r = client.query(sql).to_dataframe()
q = r.n.quantile([0.25, 0.50, 0.75, 0.99])
print(f"p25={int(q[0.25])}  median={int(q[0.5])}  p75={int(q[0.75])}  p99={int(q[0.99])}")
print(f"sparse (<10): {(r.n < 10).sum()}")
print(f"possible-dup (>3000): {(r.n > 3000).sum()}")
print(verdict(80 <= q[0.5] <= 200, warn=True))

## Test 13 — Medication Density (Polypharmacy)

In [ ]:
"""Test 13: distinct medications per admission."""
sql = f"""
SELECT hadm_id, COUNT(DISTINCT LOWER(drug)) AS n
FROM `physionet-data.mimiciv_3_1_hosp.prescriptions`
WHERE hadm_id IN ({cohort_hadm_csv()})
GROUP BY hadm_id
"""
r = client.query(sql).to_dataframe()
q = r.n.quantile([0.25, 0.50, 0.75])
print(f"p25={int(q[0.25])}  median={int(q[0.5])}  p75={int(q[0.75])}")
print(f"sparse (<3): {(r.n < 3).sum()}")
print(f"possible-dup (>100): {(r.n > 100).sum()}")
print(verdict(15 <= q[0.5] <= 30, warn=True))

## Test 14 — Vitals Density (Chartevents per ICU-day)

In [ ]:
"""Test 14: chartevents per ICU day — monitoring intensity check."""
stay_csv = ",".join(str(s) for s in df.stay_id.unique())
sql = f"""
SELECT stay_id, COUNT(*) AS n
FROM `physionet-data.mimiciv_3_1_icu.chartevents`
WHERE stay_id IN ({stay_csv}) AND valuenum IS NOT NULL
GROUP BY stay_id
"""
r = client.query(sql).to_dataframe()
r = r.merge(df[["stay_id","icu_los_days"]], on="stay_id")
r["per_day"] = r.n / r.icu_los_days.replace(0, np.nan)
q = r.per_day.quantile([0.10, 0.50, 0.90])
print(f"p10={q[0.1]:.0f}  median={q[0.5]:.0f}  p90={q[0.9]:.0f} events/day")
print(f"sparse (<20/day): {(r.per_day < 20).sum()}")
print(f"possible-dup (>3000/day): {(r.per_day > 3000).sum()}")
print(verdict(200 <= q[0.5] <= 600, warn=True))

## Test 15 — Temporal Validity of Notes (No Future Leakage)
**Critical.** A discharge note timestamped before admission or more than 24h after discharge means data linkage failure — invalidates the entire LLM evaluation.

In [ ]:
"""Test 15: discharge note charttime within admit/discharge+24h window."""
sql = f"""
SELECT COUNT(*) AS leakage
FROM `physionet-data.mimiciv_note.discharge` n
JOIN `physionet-data.mimiciv_3_1_hosp.admissions` a USING(hadm_id)
WHERE n.hadm_id IN ({cohort_hadm_csv()})
  AND (n.charttime < a.admittime OR n.charttime > TIMESTAMP_ADD(a.dischtime, INTERVAL 24 HOUR))
"""
r = client.query(sql).to_dataframe()
leaks = int(r.leakage.iloc[0])
print(f"Temporal leakage violations: {leaks}")
print(verdict(leaks == 0))

## Summary
Run all 15 tests above. A clean PASS on tests 1, 2, 4, 8, and 15 means the cohort is publication-defensible. Tests 3, 5–7, 9 should also pass; warnings are acceptable. Tests 10–14 are informational on data density and inform downstream LLM input quality.